# Pair Exercise 1 — Add Hybrid Search & a Guardrail to a Naive RAG Pipeline

**Unit 3 companion exercise — Module 1: Foundational to Enterprise RAG Architectures & Design Patterns**
**Time box: 20 minutes · Pair programming**

## How this works
- Work in **pairs**, one shared screen/notebook.
- **Driver** types; **Navigator** reads the spec, spots bugs, and asks "what does that line do?"
- **Switch roles at the 10-minute mark** (there's a checkpoint cell below to remind you).
- Don't peek at the Solution section at the bottom until you've attempted every `TODO`
  yourself — or until the facilitator calls time.

## Time plan
| Time | What to do |
|---|---|
| 0:00–2:00 | Read the starter code together. Navigator explains it back to the Driver. |
| 2:00–10:00 | **Driver A**: implement `bm25_scores()` and `fuse_scores()` (Part 1) |
| 10:00–10:30 | **Switch roles** |
| 10:30–18:00 | **Driver B**: implement `input_guardrail()` and wire it into `answer_query()` (Part 2) |
| 18:00–20:00 | Run the test cell together. Fix anything red. Discuss the stretch goal if time allows. |

## Learning objective
This exercise is the hands-on version of two Module 1 slides: **"Advanced Pattern: Hybrid
Search & Re-ranking"** and **"Guardrails & PII Protection."** By the end, you'll have a
pipeline that (a) doesn't miss exact-number queries the way pure vector search does, and
(b) refuses to process an unsafe query before it ever reaches generation.

> **Why mock embeddings?** This exercise uses small, deterministic, hash-based "embeddings"
> instead of a real model, so it runs instantly with **no internet connection and no API
> key** — nothing should get in the way of the pairing exercise itself. Demo 1
> (`Demo1_End_to_End_RAG_Pipeline.ipynb`) is where you use a real embedding model and Claude.


In [1]:
# Setup — just run this, nothing to implement here
import numpy as np
import re
from collections import Counter

print("Setup complete.")


Setup complete.


## The Knowledge Base

Same style of document set as the main Unit 3 demos: short insurance-policy clauses, each
with an ID. Notice document `D` has an exact number ("$10,000") that a meaning-only search
can miss — that's the gap Part 1 closes.


In [2]:
documents = [
    {"id": "A", "text": "Collision coverage pays for accidental physical loss to your covered auto, subject to a five hundred dollar deductible."},
    {"id": "B", "text": "Rental reimbursement covers up to forty dollars per day for a maximum of thirty days while your vehicle is being repaired."},
    {"id": "C", "text": "Water damage caused by flood or sewer backup is excluded from standard homeowners coverage."},
    {"id": "D", "text": "Any claim over $10,000 must be escalated to a Senior Claims Adjuster before settlement is authorized."},
    {"id": "E", "text": "Roadside assistance reimburses towing and labor costs up to one hundred dollars per disablement."},
]
print(f"Loaded {len(documents)} documents.")


Loaded 5 documents.


## Provided: A Toy Embedding Function (Already Implemented)

Don't change this — it's a stand-in for a real embedding model, using word hashing so it's
deterministic and needs no downloads. `vector_search()` below already works correctly; it's
your baseline.


In [3]:
DIM = 32

def fake_embed(text: str) -> np.ndarray:
    vec = np.zeros(DIM, dtype="float32")
    for w in text.lower().split():
        vec[hash(w) % DIM] += 1
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec

doc_embeddings = np.array([fake_embed(d["text"]) for d in documents])

def vector_search(query: str, k: int = 3):
    """Baseline: pure vector (cosine) similarity search. Already works — don't modify."""
    q = fake_embed(query)
    scores = doc_embeddings @ q
    top_idx = np.argsort(-scores)[:k]
    return [(documents[i], float(scores[i])) for i in top_idx]

# Try it: notice how badly vector-only search does on an exact-number query
for doc, score in vector_search("What amount requires a claim to be escalated to a Senior Claims Adjuster?"):
    print(f"[{score:.3f}] {doc['id']}: {doc['text'][:60]}...")


[0.718] E: Roadside assistance reimburses towing and labor costs up to ...
[0.702] D: Any claim over $10,000 must be escalated to a Senior Claims ...
[0.469] B: Rental reimbursement covers up to forty dollars per day for ...


---
## Part 1 (Driver A, 2:00–10:00) — Implement Hybrid Search

Vector-only search above should rank document **D** lower than it deserves for a query
about "$10,000" or "threshold," because meaning-only embeddings don't weight exact numbers
or rare keywords well. Fix this the way Module 1 taught: **fuse vector similarity with a
keyword score (BM25-style).**

### `TODO 1.1` — Implement `bm25_scores(query, corpus)`

A simplified BM25: for each document, count how many query terms it contains, weighted by
how *rare* that term is across the whole corpus (inverse document frequency). You don't need
the full BM25 formula — a reasonable approximation is enough for this exercise:

```
score(doc) = sum over query terms t in doc of (term_frequency_in_doc(t) * idf(t))
idf(t) = log(N / (1 + num_docs_containing_t))
```


In [4]:
import math

def bm25_scores(query: str, corpus_texts: list[str]) -> np.ndarray:
    """
    Return a numpy array of length len(corpus_texts) with a keyword-relevance score
    for each document, given the query.

    TODO 1.1: implement this. See the formula in the markdown cell above.
    Hint: corpus_texts[i].lower().split() gives you the words in document i.
    """
    # -------- YOUR CODE HERE --------
    query_terms = query.lower().split()
    tokenized_docs = [doc.lower().split() for doc in corpus_texts]
    n_docs = len(tokenized_docs)

    scores = np.zeros(n_docs, dtype="float32")
    for t in query_terms:
        doc_freq = sum(1 for doc in tokenized_docs if t in doc)
        idf = math.log(n_docs / (1 + doc_freq))
        for i, doc in enumerate(tokenized_docs):
            tf = doc.count(t)
            scores[i] += tf * idf
    return scores
    # ---------------------------------

# Quick sanity check while you work (not the full test suite)
test_scores = bm25_scores("threshold ten thousand dollars", [d["text"] for d in documents])
print(dict(zip([d["id"] for d in documents], np.round(test_scores, 2))))


{'A': np.float32(0.0), 'B': np.float32(0.51), 'C': np.float32(0.0), 'D': np.float32(0.0), 'E': np.float32(0.51)}


### `TODO 1.2` — Implement `fuse_scores(vector_scores, keyword_scores, vector_weight=0.6)`

Combine the two signals into one ranking. Both inputs may be on different scales, so
**normalize each to a 0–1 range before combining** (min-max normalization), then take a
weighted sum.


In [5]:
def fuse_scores(vector_scores: np.ndarray, keyword_scores: np.ndarray, vector_weight: float = 0.6) -> np.ndarray:
    """
    Combine vector and keyword scores into a single fused score per document.

    TODO 1.2: implement this.
    1. Normalize vector_scores to [0, 1] (min-max).
    2. Normalize keyword_scores to [0, 1] (min-max). Handle the all-zero case (avoid divide-by-zero).
    3. Return: vector_weight * norm_vector + (1 - vector_weight) * norm_keyword
    """
    # -------- YOUR CODE HERE --------
    def normalize(arr):
        rng = arr.max() - arr.min()
        if rng == 0:
            return np.zeros_like(arr)
        return (arr - arr.min()) / rng

    norm_vec = normalize(vector_scores)
    norm_kw = normalize(keyword_scores)
    return vector_weight * norm_vec + (1 - vector_weight) * norm_kw
    # ---------------------------------

def hybrid_search(query: str, k: int = 3, vector_weight: float = 0.6):
    q_emb = fake_embed(query)
    vec_scores = doc_embeddings @ q_emb
    kw_scores = bm25_scores(query, [d["text"] for d in documents])
    fused = fuse_scores(vec_scores, kw_scores, vector_weight=vector_weight)
    top_idx = np.argsort(-fused)[:k]
    return [(documents[i], float(fused[i])) for i in top_idx]

# Compare: document D should now rank much higher
print("-- Vector-only --")
for doc, score in vector_search("What amount requires a claim to be escalated to a Senior Claims Adjuster?"):
    print(f"[{score:.3f}] {doc['id']}")
print("\n-- Hybrid --")
for doc, score in hybrid_search("What amount requires a claim to be escalated to a Senior Claims Adjuster?"):
    print(f"[{score:.3f}] {doc['id']}")


-- Vector-only --
[0.718] E
[0.702] D
[0.469] B

-- Hybrid --
[0.972] D
[0.600] E
[0.195] B


---
## ⏸ Checkpoint (10:00) — Switch Roles Now

**Driver A → Navigator. Driver B → Driver.**

Before you continue: run the test cell in Part 1 (below) together. If any assertion fails,
that's the new Driver's first job to fix.


In [6]:
# Part 1 tests — run this before moving on
def test_part1():
    results = hybrid_search("What amount requires a claim to be escalated to a Senior Claims Adjuster?", k=1)
    top_id = results[0][0]["id"]
    assert top_id == "D", f"Expected document D to rank #1 for a claim-threshold query, got {top_id}"

    results2 = hybrid_search("collision deductible amount", k=1)
    top_id2 = results2[0][0]["id"]
    assert top_id2 == "A", f"Expected document A to rank #1 for a deductible query, got {top_id2}"

    scores = bm25_scores("flood", [d["text"] for d in documents])
    assert scores[2] > 0, "Document C mentions 'flood' — its BM25 score should be > 0"

    fused = fuse_scores(np.array([0.0, 1.0]), np.array([0.0, 0.0]))
    assert abs(fused[0]) < 1e-6 and abs(fused[1] - 0.6) < 1e-6, "fuse_scores normalization looks off"

    print("✅ Part 1 tests passed!")

test_part1()


✅ Part 1 tests passed!


---
## Part 2 (Driver B, 10:30–18:00) — Add an Input Guardrail

This is the hands-on version of the **"Guardrails & PII Protection"** slide: before any
query reaches retrieval or generation, screen it for disallowed content.

### `TODO 2.1` — Implement `input_guardrail(query)`

Return `(is_safe: bool, reason: str)`. Block a query if:
- it contains any term in `BLOCKED_TERMS` (case-insensitive), **or**
- it's longer than `MAX_QUERY_LENGTH` characters (a crude defense against prompt-injection-style walls of text).

If blocked, `reason` should briefly say why. If safe, return `(True, "")`.


In [7]:
BLOCKED_TERMS = ["ssn", "social security number", "credit card number", "ignore previous instructions"]
MAX_QUERY_LENGTH = 300

def input_guardrail(query: str) -> tuple[bool, str]:
    """
    TODO 2.1: implement this.
    Return (True, "") if the query is safe.
    Return (False, "<short reason>") if it should be blocked.
    """
    # -------- YOUR CODE HERE --------
    lowered = query.lower()
    for term in BLOCKED_TERMS:
        if term in lowered:
            return False, f"Query contains a blocked term: '{term}'"
    if len(query) > MAX_QUERY_LENGTH:
        return False, f"Query exceeds maximum length of {MAX_QUERY_LENGTH} characters"
    return True, ""
    # ---------------------------------

print(input_guardrail("What is my deductible?"))
print(input_guardrail("Please give me the credit card number on file"))


(True, '')
(False, "Query contains a blocked term: 'credit card number'")


### `TODO 2.2` — Wire the Guardrail into `answer_query()`

Complete the function below so that:
1. It calls `input_guardrail()` first.
2. If unsafe, it returns a refusal message that includes the reason — **and does not call
   `hybrid_search` at all** (the whole point of an input guardrail is to short-circuit
   before any retrieval or generation happens).
3. If safe, it runs `hybrid_search` and returns a simple formatted answer citing the top
   document's ID (no real LLM call needed for this exercise).


In [8]:
def answer_query(query: str, k: int = 2) -> str:
    """
    TODO 2.2: implement this using input_guardrail() and hybrid_search().
    """
    # -------- YOUR CODE HERE --------
    is_safe, reason = input_guardrail(query)
    if not is_safe:
        return f"Request blocked by input guardrail: {reason}"

    results = hybrid_search(query, k=k)
    top_doc, top_score = results[0]
    return f"[Answer grounded in {top_doc['id']}] {top_doc['text']}"
    # ---------------------------------

print(answer_query("What is the rental reimbursement daily limit?"))
print(answer_query("What is my social security number on file?"))


[Answer grounded in B] Rental reimbursement covers up to forty dollars per day for a maximum of thirty days while your vehicle is being repaired.
Request blocked by input guardrail: Query contains a blocked term: 'social security number'


In [9]:
# Part 2 tests — run this before the final checkpoint
def test_part2():
    safe, reason = input_guardrail("What is my deductible?")
    assert safe is True and reason == "", "A normal query should pass the guardrail"

    unsafe, reason = input_guardrail("What is the credit card number on file?")
    assert unsafe is False and len(reason) > 0, "A blocked-term query should fail with a reason"

    long_query = "a" * 400
    unsafe_long, _ = input_guardrail(long_query)
    assert unsafe_long is False, "An overly long query should be blocked"

    blocked_answer = answer_query("What is the social security number on file?")
    assert "blocked" in blocked_answer.lower(), "answer_query should return a refusal message for unsafe input"
    assert "[Answer grounded in" not in blocked_answer, "answer_query must NOT retrieve/generate for a blocked query"

    safe_answer = answer_query("What is the rental reimbursement daily limit?")
    assert "[Answer grounded in B]" in safe_answer, "A safe query about rental reimbursement should cite document B"

    print("✅ Part 2 tests passed!")

test_part2()


✅ Part 2 tests passed!


---
## Wrap-Up (18:00–20:00)

Run both test cells one more time together. Then discuss as a pair:
- What would happen if you swapped the order — ran retrieval *before* the guardrail check?
  Why does Module 1 insist guardrails come first?
- Your `bm25_scores` is a simplification. What's one real BM25 detail you skipped (hint:
  document length normalization)?

### Stretch Goal (if you finish early)
Add a **second guardrail** — an *output* guardrail. Write `output_guardrail(answer: str)`
that blocks the answer if it contains a dollar amount **not present in the source
document's text** (a crude hallucination check). Wire it into `answer_query()` after
generation. This mirrors the "Guardrails should be layered" point from the deck: input
guardrails and output guardrails catch different failure modes.


---
---
# Solution (Facilitator Reference — reveal only after the exercise)

The `TODO` cells above already contain a complete, working reference solution inline
(between the `-------- YOUR CODE HERE --------` markers), so pairs who want to compare their
approach can look there directly once time is called. Common alternative-but-valid
approaches to discuss:

- **Part 1:** some pairs will implement BM25 scoring using raw term counts without IDF
  weighting. This still passes the tests here (the corpus is tiny) but would perform worse
  on a larger, more realistic corpus — worth raising as a discussion point.
- **Part 1 fusion:** some pairs may normalize by dividing by the max only (not min-max).
  This also passes if all scores are non-negative, but min-max is more robust in general.
- **Part 2:** some pairs may check length before blocked terms, or combine both checks into
  one `if`. Order doesn't matter functionally here, only that both checks happen before any
  retrieval call.
